# Prosjekt 2

Info

In [16]:
# Necessary libraries
import scipy
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
import jax.lax as lax
import time
import numpy as np
from jax.experimental.ode import odeint
from functools import partial

jax.config.update("jax_enable_x64", True)

## Model Problems

In [17]:
def r(x: jax.Array) -> float:
    return jnp.sqrt(x[0]**2+ x[1]**2)

def r2(x: jax.Array) -> float:
    return x[0]**2+ x[1]**2

#### Charged Particle

In [18]:
def B_cp(x: jax.Array) -> jax.Array:
    return jnp.array([0, 0, r(x)])


def A_cp(x: jax.Array) -> jax.Array:
    return 1/3 * jnp.array([-x[1]*r(x), x[0]*r(x), 0])


def U_cp(x: jax.Array) -> float:
    rr = jnp.clip(r(x), 1e-20, None) # To make sure the I dont get divide by zero errors
    return 1 / (100 * rr)

def gradU_cp(x: jax.Array) -> jax.Array:
    rr= jnp.clip(r(x), 1e-20, None) # TO make sure the I dont get divide by zero errors
    return -1/ 100 *jnp.array([x[0]/rr**3, x[1]/rr**3, 0])

# System of differential equations
@jax.jit
def system_cp(t: jax.Array, Y: jax.Array) -> jax.Array:
    x = Y[:3]
    v = Y[3:]
    dxdt = v
    dvdt = jnp.cross(v, B_cp(x)) - gradU_cp(x)
    return jnp.concatenate((dxdt, dvdt))

charged_particle = dict(
    name = "charged particle",
    x0 = jnp.array([0, 1, 0.1]),
    v0 = jnp.array([0.09, 0.05, 0.20]),
    B = B_cp,
    A = A_cp,
    U = U_cp,
    gradU = gradU_cp,
    system = system_cp
)

#### Tokamak

In [19]:
def B_tok(x: jax.Array) -> jax.Array:
    """
    Returns the forces of the magnetic Field
    """

    rr2 = jnp.maximum(r2(x), 1e-20) # TO make sure the I dont get divide by zero errors
    rr = jnp.maximum(r(x), 1e-20) # TO make sure the I dont get divide by zero errors

    return jnp.array([-(2 * x[1] + x[0]*x[2]) / (2 *rr2),
                          (2*x[0] - x[1]*x[2]) / (2* rr2),
                                    (r(x) -1) / (2* rr)])


def A_tok(x: jax.Array) -> jax.Array:

    rr2 = jnp.maximum(r2(x), 1e-20) # TO make sure the I dont get divide by zero errors
    rr = jnp.maximum(r(x), 1e-20) # TO make sure the I dont get divide by zero errors

    return jnp.array([0, 
                     1/2  * ( x[0] - jnp.log(rr + x[0]) ), 
                     -1/2 * (x[2] * jnp.arctan(x[1]/x[0]) + jnp.log(rr2)) ])

def U_tok(x: jax.Array) -> float:
    return 0.0

def gradU_tok(x: jax.Array) -> jax.Array:
    return jnp.array([0.0, 0.0, 0.0])


# # System of differential equations
@jax.jit
def system_tok(t: float, Y: jax.Array) -> jax.Array:
    x = Y[:3]
    v = Y[3:]
    dxdt = v
    dvdt = jnp.cross(v, B_tok(x)) - gradU_tok(x)
    return jnp.concatenate((dxdt, dvdt))

tokamak = dict(
    name = "tokamak",
    x0 = jnp.array([1.2, 0.0, 0.0]),
    v0 = jnp.array([0.0, 4.816e-4, -2.059e-3]),
    B = B_tok,
    A = A_tok,
    U = U_tok,
    gradU = gradU_tok,
    system = system_tok
)

## Helper functions

In [20]:
def energy(problem, x, v):
    U = jax.vmap(problem["U"], in_axes=1)
    return 0.5 * jnp.sum(v**2, axis=0) + U(x)


def reference_solution(problem, t_eval):
    Y0 = jnp.concatenate([problem["x0"], problem["v0"]])

    #jax odeint expects system to have arg (Y, t) instead of (t, Y)
    def f(Y, t):
        return problem["system"](t, Y)

    Y = odeint(f, Y0, jnp.asarray(t_eval), rtol=1e-12, atol=1e-12).T

    return t_eval, Y[:3], Y[3:]

def timed(solve, problem, h, T):

    #lets jit warm up one round before measuring the actual compute time
    jax.block_until_ready(solve(problem, h, T))
    
    start = time.perf_counter()
    out = jax.block_until_ready(solve(problem, h, T))

    return out, time.perf_counter() - start

## Solvers

### Boris

### RK4

In [21]:
def RK4_step(t:jax.Array, Yn:jax.Array, system:jax.Array, h:float) ->jax.Array:
    """
    t: time (is ignored here)
    Yn: [x1,x2,x3,v1,v2,v3]
    system: gives dY/dt (RHS of differential equation)
    h: steplength
    returns: Yn_new

    """
    k1 = h * system(t, Yn)
    k2 = h * system(t, Yn + 0.5*k1)
    k3 = h * system(t, Yn + 0.5*k2)
    k4 = h * system(t, Yn + k3)

    Yn_new = Yn + 1/6*(k1 + 2*k2 + 2*k3 + k4)

    return Yn_new

@partial(jax.jit, static_argnames=("system", "N"))
def rk4_subwrap(N: int,  Y0:jax.Array, system, h:float) ->tuple[jax.Array, jax.Array]:
    """
    N: number of evaluation pts
    Y0: initial Y vector
    system: RHS differential eqt
    h: step length
    """
    def body(Yn, _):
        Yn_new = RK4_step(0.0, Yn, system, h)
        return Yn_new, Yn_new

    Y_final, trajectory = lax.scan(body, Y0, xs=None, length=N)
    
    Y = jnp.concatenate((Y0[None, :], trajectory), axis=0).T

    return Y

def rk4(problem, h, T):
    N = int(round(T / h))
    Y0 = jnp.concatenate([problem["x0"], problem["v0"]])
    Y = rk4_subwrap(N, Y0, problem["system"], h)
    t = h * jnp.arange(N+1)
    return t, Y[:3], Y[3:]

### Multistep